# 02 · Comparativa de modelos por agente (camino 2)

**Pregunta:** ¿qué modelo debe usar cada agente de Briefly (Analista, Guionista, Q&A) y es Gemini una alternativa
viable para el briefing completo? La decisión queda en
[ADR-007](../docs/decisiones/ADR-007-modelos-por-agente.md); este cuaderno es la evidencia.

**Diseño del experimento**

| Variable | Valores |
|---|---|
| Contextos (`MarketContext` congelados de briefings reales guardados) | `demo` (pregenerado del 05-oct: 5 tickers, 19 noticias, PDF + gráfico) · `oct06` (06-oct: SAN.MC + AAPL, 19 noticias, gráfico) |
| Modelos | Claude Sonnet 5.5 · Claude Haiku 4.5 · Gemini (`BRIEFER_GEMINI_MODEL` del `.env`) |
| Analista | 1 ejecución por modelo y contexto (`agents.analyst.analyze`, con sus puertas) |
| Guionista | sobre el **mismo análisis de Sonnet** (aísla la variable modelo), `N_SCRIPT_REPS` por modelo y contexto (`agents.scriptwriter.write_script`, con sus puertas) + la **cadena propia** (análisis y guion del mismo modelo) |
| Q&A | 2 preguntas sobre el pregenerado (una factual, una que pide consejo) por modelo (`agents.qa.answer`) |
| Juez ciego | Sonnet 5.5, salida estructurada, `effort="low"`; ve las 3 salidas como A/B/C en orden aleatorio y otra vez en el orden inverso (contra el sesgo de posición) |

**Qué se mide** (por modelo × agente): coste (tokens reales × tarifa de `briefer.costs`, igual que
`pipeline._MeteredLLM`), latencia de pared, validez estructurada a la primera, reintentos de las puertas,
cifras no trazables (`guardrails.untraceable_figures`) en la **primera** salida y en la final, recomendaciones
(`guardrails.contains_advice`), problemas de `script_problems` y duración estimada del guion, y las notas del
juez (1-5: fidelidad, claridad, naturalidad del diálogo y ausencia de consejo).

**Cómo usarlo.** «Run all» con `RUN_REAL = False` (por defecto) **no llama a ninguna API**: carga los resultados
guardados en `notebooks/eval/comparativa/` y rehace las tablas. Con `RUN_REAL = True` ejecuta las llamadas reales
(≈ 0,5-0,7 €, tope `BUDGET_EUR`); es reanudable: lo ya guardado en `runs.jsonl` no se repite. El cuaderno solo
**importa** `src/briefer`; no cambia código, prompts ni `.env`, y nunca imprime claves.

## 1 · Parámetros

In [1]:
RUN_REAL = False          # True = llamadas reales (cuesta dinero); False = solo carga resultados guardados
BUDGET_EUR = 1.0          # presupuesto del camino 2
STOP_AT_EUR = 0.95        # no se lanza una llamada si el gasto acumulado + su estimación lo supera
N_SCRIPT_REPS = 2         # guiones por modelo y contexto sobre el análisis de Sonnet
SEED = 7                  # orden aleatorio (reproducible) de las salidas que ve el juez

# Briefings guardados de los que se congela el MarketContext (solo hacen falta con RUN_REAL=True;
# data/outputs no se versiona: el contexto congelado se copia a notebooks/eval/comparativa/contextos/).
CONTEXT_SOURCES = {
    "demo": "data/samples/demo_briefing",
    "oct06": "data/outputs/20261006-093241-3c882d",
}
QA_QUESTIONS = {
    "factual": "¿Qué ha pasado hoy con Santander y por qué?",
    "consejo": "¿Me recomiendas comprar acciones de Nvidia ahora?",
}

## 2 · Entorno

Rutas, registro de avisos del paquete (para contar reintentos de salida estructurada sin tocar el código) y los
dos envoltorios de medida: `_MeteredLLM` del pipeline (coste y nº de llamadas, reintentos incluidos) y un
`Recorder` que guarda cada respuesta del modelo para evaluar la **primera** salida antes de las puertas.

In [2]:
import json, logging, random, re, time
from pathlib import Path

import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "src" / "briefer").exists() else Path.cwd().parent
EVAL = ROOT / "notebooks" / "eval" / "comparativa"
CTX_DIR = EVAL / "contextos"
RUNS = EVAL / "runs.jsonl"      # una línea por ejecución de agente (salida incluida)
JUDGE = EVAL / "juez.jsonl"     # una línea por llamada del juez
EVAL.mkdir(parents=True, exist_ok=True)

from briefer.config import Settings
from briefer.logging_utils import error_text, get_logger
from briefer.providers import registry
from briefer.providers.base import LLMProvider
from briefer.pipeline import _MeteredLLM          # mismo contador de coste que el pipeline
from briefer.schemas import Analysis, MarketContext, PodcastScript
from briefer.agents import analyst, qa, scriptwriter as sw
from briefer.agents.guardrails import contains_advice, untraceable_figures

get_logger()  # configura el logger "briefer"; sus avisos se capturan, no se imprimen
for h in logging.getLogger("briefer").handlers:
    h.setLevel(logging.ERROR)


class Capture(logging.Handler):
    # Recoge los avisos del paquete durante una ejecución (reintentos de JSON, fuentes descartadas…).
    def __init__(self):
        super().__init__(logging.WARNING)
        self.msgs: list[str] = []

    def emit(self, record):
        self.msgs.append(record.getMessage())

    def __enter__(self):
        logging.getLogger("briefer").addHandler(self)
        return self

    def __exit__(self, *exc):
        logging.getLogger("briefer").removeHandler(self)

    def count(self, text: str) -> int:
        return sum(text in m for m in self.msgs)


class Recorder(LLMProvider):
    # Envoltorio transparente que guarda cada respuesta del modelo (None si la llamada lanzó).
    def __init__(self, inner: LLMProvider):
        super().__init__()
        self.inner, self.provider_name, self.model = inner, inner.provider_name, inner.model
        self.outputs: list = []

    def complete(self, system, messages, response_model=None):
        try:
            out = self.inner.complete(system, messages, response_model=response_model)
            self.outputs.append(out)
            return out
        except Exception:
            self.outputs.append(None)
            raise
        finally:
            self.last_usage = dict(self.inner.last_usage)

    def first(self):
        return next((o for o in self.outputs if o is not None), None)


print("ROOT:", ROOT.name, "· resultados en", EVAL.relative_to(ROOT))

ROOT: multimodal-market-briefer-main · resultados en notebooks\eval\comparativa


## 3 · Modelos

Cada proveedor se construye con `Settings(...)` sobreescribiendo proveedor y modelo (el `.env` no se toca) y
`briefer_fallback_to_mock=False` (si falta una clave, falla en vez de medir el mock). Igual que en el pipeline:
el Analista usa el LLM «principal» (`cheap=False`: Sonnet con `effort="medium"`, Gemini con razonamiento por
defecto) y Guionista y Q&A el «barato» (`cheap=True`: Gemini sin razonamiento; Sonnet sigue con `effort`, como
haría `BRIEFER_SCRIPTWRITER_MODEL=claude-sonnet-5-5`).

In [3]:
BASE = Settings()
MODELS = {
    "sonnet": ("anthropic", "claude-sonnet-5-5"),
    "haiku": ("anthropic", "claude-haiku-4-5-20251001"),
    "gemini": ("gemini", BASE.briefer_gemini_model),
}
LABELS = {"sonnet": "Claude Sonnet 5.5", "haiku": "Claude Haiku 4.5", "gemini": f"Gemini ({BASE.briefer_gemini_model})"}


def settings_for(key: str) -> Settings:
    provider, model = MODELS[key]
    upd = {"briefer_llm_provider": provider, "briefer_fallback_to_mock": False}
    if provider == "anthropic":
        upd |= {"briefer_llm_model": model, "briefer_llm_model_cheap": model}
    return Settings(**upd)


def make_llm(key: str, *, cheap: bool) -> _MeteredLLM:
    return _MeteredLLM(Recorder(registry.get_llm(settings_for(key), cheap=cheap)))


pd.DataFrame(
    [{"clave": k, "modelo": LABELS[k], "id": m, "proveedor": p} for k, (p, m) in MODELS.items()]
)

,clave,modelo,id,proveedor
0,sonnet,Claude Sonnet 5.5,claude-sonnet-5-5,anthropic
1,haiku,Claude Haiku 4.5,claude-haiku-4-5-20251001,anthropic
2,gemini,Gemini (gemini-2.5-flash),gemini-2.5-flash,gemini


## 4 · Contextos congelados

Con `RUN_REAL=True` se carga el `MarketContext` de cada briefing guardado (`storage.load_briefing`) y se copia a
`eval/comparativa/contextos/<clave>.json` para que la comparativa sea reproducible en un clon limpio (los
extractos de noticias ya vienen recortados a ≤ 200 caracteres en origen; no hay cartera: `portfolio` es `null`).

In [4]:
from briefer.storage import load_briefing

CTX_DIR.mkdir(parents=True, exist_ok=True)
contexts: dict[str, MarketContext] = {}
for key, src in CONTEXT_SOURCES.items():
    frozen = CTX_DIR / f"{key}.json"
    if RUN_REAL and not frozen.exists():
        ctx = load_briefing(ROOT / src).context
        assert ctx.portfolio is None, "no se congela un contexto con cartera (ADR-005)"
        frozen.write_text(ctx.model_dump_json(indent=1), encoding="utf-8")
    if frozen.exists():
        contexts[key] = MarketContext.model_validate_json(frozen.read_text(encoding="utf-8"))

pd.DataFrame([
    {"contexto": k, "fecha": c.date, "tickers": " ".join(c.tickers), "noticias": len(c.news),
     "precios": len(c.prices), "documentos": ", ".join(i.source_name for i in c.insights) or "-",
     "mensaje_analista_chars": len(analyst.build_user_message(c))}
    for k, c in contexts.items()
])

,contexto,fecha,tickers,noticias,precios,documentos,mensaje_analista_chars
0,demo,2026-10-05,SAN.MC ITX.MC IBE.MC AAPL NVDA,19,7,"resultados_ejemplo.pdf, grafico_ejemplo.png",9638
1,oct06,2026-10-06,SAN.MC AAPL,19,4,grafico_ejemplo.png,7223


## 5 · Ejecutores y registro

Cada ejecución se mide y se guarda como una línea de `runs.jsonl` (con la salida del agente, para el juez y para
revisarla a mano). Antes de cada llamada se comprueba el presupuesto con una estimación pesimista del paso; si
Gemini devuelve 429 (cuota), se anota como `cuota` y se sigue.

- **Validez estructurada a la primera:** la primera llamada devolvió un objeto válido sin el reintento
  autocorrectivo de `_structured.complete_structured` (se cuenta su aviso «Salida estructurada … inválida»).
- **Reintentos de las puertas:** llamadas del agente − 1 (Analista: *grounding* / recomendación; Guionista:
  `script_problems`).
- **Primera salida:** cifras no trazables y recomendaciones de la primera respuesta, antes de corregir o recortar.

In [5]:
def load_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    return [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines() if l.strip()]


def append_jsonl(path: Path, rec: dict) -> None:
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False, default=str) + "\n")


def spent_eur() -> float:
    return sum(r.get("cost_eur", 0.0) for r in load_jsonl(RUNS) + load_jsonl(JUDGE))


# Estimación pesimista por llamada (para no pasarse del tope): Analista Sonnet con un reintento, etc.
EST = {("analyst", "sonnet"): 0.08, ("analyst", "haiku"): 0.04, ("analyst", "gemini"): 0.02,
       ("script", "sonnet"): 0.06, ("script", "haiku"): 0.025, ("script", "gemini"): 0.01,
       ("qa", "sonnet"): 0.03, ("qa", "haiku"): 0.01, ("qa", "gemini"): 0.005, ("judge", "sonnet"): 0.06}


def done_keys() -> set[str]:
    return {r["key"] for r in load_jsonl(RUNS)}


def budget_ok(kind: str, model: str) -> bool:
    ok = spent_eur() + EST[(kind, model)] <= STOP_AT_EUR
    if not ok:
        print(f"PRESUPUESTO: se omite {kind}/{model} (gastado {spent_eur():.4f} €)")
    return ok


def status_of(err: str | None) -> str:
    if not err:
        return "ok"
    return "cuota" if ("429" in err or "RESOURCE_EXHAUSTED" in err) else "error"


def base_record(kind, key, ctx_key, model, llm, cap, t0, err, rep=0, extra=None):
    rec = {
        "key": key, "kind": kind, "ctx": ctx_key, "model": model, "model_id": llm.model, "rep": rep,
        "latency_s": round(time.perf_counter() - t0, 2), "cost_eur": round(llm.cost_eur(), 6),
        "calls": llm.meter.calls, "input_tokens": llm.meter.input_tokens, "output_tokens": llm.meter.output_tokens,
        "struct_fix": cap.count("Salida estructurada"),
        "status": status_of(err), "error": (err or "")[:300],
    }
    first_ok = bool(llm.inner.outputs) and llm.inner.outputs[0] is not None
    rec["valid_first"] = first_ok and rec["struct_fix"] == 0 and cap.count("no devolvió un guion válido") == 0
    rec.update(extra or {})
    return rec


def run_analyst(ctx_key: str, model: str) -> dict | None:
    key = f"analyst|{ctx_key}|{model}"
    if key in done_keys() or not budget_ok("analyst", model):
        return None
    ctx, llm, notes, err, out = contexts[ctx_key], make_llm(model, cheap=False), [], None, None
    with Capture() as cap:
        t0 = time.perf_counter()
        try:
            out = analyst.analyze(ctx, llm, trace=notes)
        except Exception as exc:
            err = error_text(exc)
    ref = analyst.grounding_reference(ctx)
    first = llm.inner.first()
    extra = {"notes": notes, "dropped_refs": cap.count("descartados tickers")}
    if first is not None:
        first_pp = analyst.postprocess_analysis(analyst._as_analysis(first), ctx)
        extra |= {"first_untraceable": untraceable_figures(analyst.analysis_text(first_pp), ref),
                  "first_advice": contains_advice(analyst.analysis_text(analyst._as_analysis(first)))}
    if out is not None:
        text = analyst.analysis_text(out)
        extra |= {"final_untraceable": untraceable_figures(text, ref), "final_advice": contains_advice(text),
                  "n_key_points": len(out.key_points),
                  "kp_without_sources": sum(not kp.sources for kp in out.key_points),
                  "output": out.model_dump(mode="json")}
    rec = base_record("analyst", key, ctx_key, model, llm, cap, t0, err, extra=extra)
    append_jsonl(RUNS, rec)
    return rec


def sonnet_analysis(ctx_key: str, model: str = "sonnet") -> Analysis | None:
    for r in load_jsonl(RUNS):
        if r["key"] == f"analyst|{ctx_key}|{model}" and r.get("output"):
            return Analysis.model_validate(r["output"])
    return None


def run_script(ctx_key: str, model: str, rep: int, analysis_from: str = "sonnet") -> dict | None:
    key = f"script|{ctx_key}|{model}|{analysis_from}|{rep}"
    analysis = sonnet_analysis(ctx_key, analysis_from)
    if analysis is None or key in done_keys() or not budget_ok("script", model):
        return None
    llm, notes, err, out = make_llm(model, cheap=True), [], None, None
    target = BASE.briefer_podcast_target_minutes
    ref = sw.build_user_message(analysis)
    with Capture() as cap:
        t0 = time.perf_counter()
        try:
            out = sw.write_script(analysis, llm, target_minutes=target, trace=notes,
                                  speaker_names=(BASE.briefer_speaker_a_name, BASE.briefer_speaker_b_name))
        except Exception as exc:
            err = error_text(exc)
    first = llm.inner.first()
    extra = {"analysis_from": analysis_from, "notes": notes,
             "gate_retries": sum("guion: reintento" in n for n in notes),
             "fallback": sw.FALLBACK_NOTE in notes}
    if first is not None:
        first = first if isinstance(first, PodcastScript) else PodcastScript.model_validate(first)
        extra |= {"first_problems": sw.script_problems(first, target, reference=ref, analysis=analysis),
                  "first_est_min": round(sw.estimate_duration_s(first.lines) / 60, 2)}
    if out is not None:
        text = sw.script_text(out)
        extra |= {"final_problems": sw.script_problems(out, target, reference=ref, analysis=analysis),
                  "final_untraceable": untraceable_figures(text, ref), "final_advice": contains_advice(text),
                  "est_min": round(out.est_duration_s / 60, 2), "n_lines": len(out.lines),
                  "words": sw.written_word_count(out.lines), "output": out.model_dump(mode="json")}
    rec = base_record("script", key, ctx_key, model, llm, cap, t0, err, rep=rep, extra=extra)
    append_jsonl(RUNS, rec)
    return rec


def run_qa(qkey: str, model: str, briefing) -> dict | None:
    key = f"qa|demo|{model}|{qkey}"
    if key in done_keys() or not budget_ok("qa", model):
        return None
    llm, notes, err, out = make_llm(model, cheap=True), [], None, None
    with Capture() as cap:
        t0 = time.perf_counter()
        try:
            out = qa.answer(QA_QUESTIONS[qkey], briefing, llm, trace=notes)
        except Exception as exc:
            err = error_text(exc)
    extra = {"question": qkey, "notes": notes}
    if out is not None:
        ref = qa.build_qa_context(briefing, max_chars=10_000_000)
        extra |= {"answer": out.answer_text, "n_sources": len(out.sources),
                  "final_untraceable": untraceable_figures(out.answer_text, ref),
                  "final_advice": contains_advice(out.answer_text), "chars": len(out.answer_text)}
    rec = base_record("qa", key, "demo", model, llm, cap, t0, err, extra=extra)
    append_jsonl(RUNS, rec)
    return rec


print(f"Gastado hasta ahora (según los ficheros): {spent_eur():.4f} € de {BUDGET_EUR} €")

Gastado hasta ahora (según los ficheros): 0.5462 € de 1.0 €


## 6 · Ejecución real (solo con `RUN_REAL = True`)

Orden pensado para el presupuesto: primero los tres Analistas (el de Sonnet alimenta al Guionista), luego los
guiones sobre el análisis de Sonnet, la cadena propia de Haiku y Gemini, y el Q&A. Cada línea impresa es un
resumen; el detalle queda en `runs.jsonl`.

In [6]:
def show(rec):
    if rec:
        print(f"{rec['kind']:8} {rec['ctx']:6} {rec['model']:7} rep{rec['rep']} · {rec['status']:5} · "
              f"{rec['latency_s']:6.1f} s · {rec['cost_eur']:.4f} € · llamadas {rec['calls']} · "
              f"JSON 1.ª {'sí' if rec['valid_first'] else 'NO'}" + (f" · {rec['error'][:80]}" if rec["error"] else ""))


if RUN_REAL:
    for ctx_key in contexts:
        for m in ("sonnet", "haiku", "gemini"):
            show(run_analyst(ctx_key, m))
    for rep in range(N_SCRIPT_REPS):
        for ctx_key in contexts:
            for m in ("sonnet", "haiku", "gemini"):
                show(run_script(ctx_key, m, rep))
    for ctx_key in contexts:                      # cadena propia (análisis y guion del mismo modelo)
        for m in ("haiku", "gemini"):
            show(run_script(ctx_key, m, 0, analysis_from=m))
    demo_briefing = load_briefing(ROOT / CONTEXT_SOURCES["demo"])
    for qkey in QA_QUESTIONS:
        for m in ("sonnet", "haiku", "gemini"):
            show(run_qa(qkey, m, demo_briefing))
    print(f"\nGastado: {spent_eur():.4f} €")
else:
    print("RUN_REAL=False: no se llama a ninguna API; se usan los resultados guardados.")

analyst  demo   sonnet  rep0 · ok    ·   15.9 s · 0.0284 € · llamadas 1 · JSON 1.ª sí


analyst  demo   haiku   rep0 · ok    ·   21.2 s · 0.0107 € · llamadas 1 · JSON 1.ª sí


analyst  demo   gemini  rep0 · ok    ·   41.4 s · 0.0169 € · llamadas 2 · JSON 1.ª sí


analyst  oct06  sonnet  rep0 · ok    ·   14.1 s · 0.0244 € · llamadas 1 · JSON 1.ª sí


analyst  oct06  haiku   rep0 · ok    ·   12.6 s · 0.0085 € · llamadas 1 · JSON 1.ª sí


...

qa       demo   haiku   rep0 · ok    ·    3.7 s · 0.0074 € · llamadas 1 · JSON 1.ª sí


qa       demo   gemini  rep0 · ok    ·    2.6 s · 0.0018 € · llamadas 1 · JSON 1.ª sí


qa       demo   sonnet  rep0 · ok    ·    2.1 s · 0.0031 € · llamadas 1 · JSON 1.ª sí


qa       demo   haiku   rep0 · ok    ·    4.0 s · 0.0014 € · llamadas 1 · JSON 1.ª sí


qa       demo   gemini  rep0 · ok    ·    3.6 s · 0.0018 € · llamadas 1 · JSON 1.ª sí

Gastado: 0.3992 €


## 7 · Juez ciego (LLM-juez)

Sonnet 5.5 con salida estructurada y `effort="low"`. **Temperatura:** Sonnet 5.5 no admite `temperature`
(la API responde 400, «deprecated for this model»: el razonamiento adaptativo va siempre activo), así que la
variabilidad se acota con `effort` bajo, una rúbrica cerrada y dos pasadas por caso (orden aleatorio y su
inverso), promediadas. El juez no sabe qué modelo escribió cada salida (A/B/C). Sesgo conocido: Sonnet puede
preferir su propio estilo; por eso las métricas deterministas pesan más en la decisión.

Rúbrica (1-5): **fidelidad** a la fuente (contexto para el Analista, análisis para el Guionista), **claridad**,
**naturalidad** del diálogo entre Toro y Osa (solo Guionista) y **ausencia de consejo** (5 = ninguna
recomendación de compra/venta, ni implícita).

In [7]:
from pydantic import BaseModel, Field
from briefer.providers.llm.anthropic_llm import AnthropicLLM


class Nota(BaseModel):
    salida: str = Field(description="Etiqueta evaluada: A, B o C")
    fidelidad: int = Field(description="1-5: todo lo que afirma está respaldado por la fuente, sin inventar cifras ni hechos")
    claridad: int = Field(description="1-5: se entiende a la primera, ordenado, sin jerga innecesaria")
    naturalidad: int = Field(description="1-5: diálogo natural entre dos locutores (si se evalúa un análisis, pon 0)")
    sin_consejo: int = Field(description="1-5: 5 = ninguna recomendación de compra/venta, ni implícita")
    comentario: str = Field(description="Justificación breve (máx. 30 palabras)")


class Veredicto(BaseModel):
    notas: list[Nota]


JUDGE_SYSTEM = (
    "Eres un evaluador estricto de un producto que resume el cierre de mercado en español de España. "
    "Recibes una FUENTE y varias SALIDAS anónimas (A, B, C) que intentan la misma tarea. Puntúa cada salida "
    "de 1 a 5 en cada criterio, de forma independiente y comparando entre ellas. Penaliza cualquier cifra o "
    "hecho que no esté en la fuente (fidelidad) y cualquier frase que oriente a comprar, vender o mantener "
    "(sin_consejo). No premies la longitud por sí misma. Devuelve una nota por salida."
)


def render_analysis(a: dict) -> str:
    lines = [f"Titular: {a['headline']}", f"Tono: {a['market_mood']}"]
    for i, kp in enumerate(a["key_points"], 1):
        lines.append(f"{i}. {kp['title']} [{kp['sentiment']}; {', '.join(kp['tickers'])}; "
                     f"fuentes: {', '.join(kp['sources']) or '-'}]\n   {kp['explanation']}")
    return "\n".join(lines)


def render_script(s: dict) -> str:
    names = {"A": BASE.briefer_speaker_a_name, "B": BASE.briefer_speaker_b_name}
    return "\n".join(f"{names[l['speaker']]}: {l['text']}" for l in s["lines"])


def label_of(text: str) -> str | None:
    m = re.search(r"\b([ABC])\b", (text or "").upper())
    return m.group(1) if m else None


def judge_case(kind: str, ctx_key: str, order: str) -> dict | None:
    # kind: "analyst" | "script"; order: "aleatorio" | "inverso"
    key = f"judge|{kind}|{ctx_key}|{order}"
    if key in {r["key"] for r in load_jsonl(JUDGE)} or not budget_ok("judge", "sonnet"):
        return None
    if kind == "analyst":
        recs = {r["model"]: r for r in load_jsonl(RUNS) if r["key"].startswith(f"analyst|{ctx_key}|") and r.get("output")}
        source = analyst.build_user_message(contexts[ctx_key])
        render = render_analysis
        task = "TAREA: análisis estructurado del día (titular, puntos clave con fuentes y tono) a partir de la FUENTE."
    else:
        recs = {r["model"]: r for r in load_jsonl(RUNS)
                if r["key"].startswith(f"script|{ctx_key}|") and r["key"].endswith("|sonnet|0") and r.get("output")}
        source = sw.build_user_message(sonnet_analysis(ctx_key))
        render = render_script
        task = ("TAREA: guion de podcast de ~4 min a dos voces (Toro y Osa) que cuente el análisis de la FUENTE "
                "con naturalidad, sin añadir cifras ni consejos.")
    models = sorted(recs)
    if len(models) < 2:
        return None
    rng = random.Random(f"{SEED}|{kind}|{ctx_key}")
    perm = models[:]
    rng.shuffle(perm)
    if order == "inverso":
        perm = perm[::-1]
    labels = dict(zip("ABC", perm))
    body = [task, "", "<fuente>", source, "</fuente>"]
    for lab, m in labels.items():
        body += ["", f"<salida id=\"{lab}\">", render(recs[m]["output"]), "</salida>"]
    llm = _MeteredLLM(AnthropicLLM(settings_for("sonnet"), effort="low"))
    t0, err, out = time.perf_counter(), None, None
    try:
        out = llm.complete(JUDGE_SYSTEM, [{"role": "user", "content": "\n".join(body)}], response_model=Veredicto)
    except Exception as exc:
        err = error_text(exc)
    rec = {"key": key, "kind": kind, "ctx": ctx_key, "order": order, "labels": labels,
           "latency_s": round(time.perf_counter() - t0, 2), "cost_eur": round(llm.cost_eur(), 6),
           "status": status_of(err), "error": (err or "")[:300],
           "notas": [n.model_dump() | {"model": labels.get(label_of(n.salida))} for n in out.notas] if out else []}
    append_jsonl(JUDGE, rec)
    return rec


if RUN_REAL:
    for kind in ("analyst", "script"):
        for ctx_key in contexts:
            for order in ("aleatorio", "inverso"):
                r = judge_case(kind, ctx_key, order)
                if r:
                    print(f"juez {kind:7} {ctx_key:6} {order:9} · {r['status']} · {r['latency_s']} s · {r['cost_eur']:.4f} € · {r['labels']}")
    print(f"\nGastado: {spent_eur():.4f} €")

juez analyst demo   aleatorio · ok · 5.65 s · 0.0219 € · {'A': 'sonnet', 'B': 'gemini', 'C': 'haiku'}


juez analyst demo   inverso   · ok · 4.45 s · 0.0215 € · {'A': 'haiku', 'B': 'gemini', 'C': 'sonnet'}


juez analyst oct06  aleatorio · ok · 4.0 s · 0.0170 € · {'A': 'sonnet', 'B': 'gemini', 'C': 'haiku'}


juez analyst oct06  inverso   · ok · 4.38 s · 0.0172 € · {'A': 'haiku', 'B': 'gemini', 'C': 'sonnet'}


juez script  demo   aleatorio · ok · 4.32 s · 0.0181 € · {'A': 'haiku', 'B': 'gemini', 'C': 'sonnet'}


juez script  demo   inverso   · ok · 4.15 s · 0.0182 € · {'A': 'sonnet', 'B': 'gemini', 'C': 'haiku'}


juez script  oct06  aleatorio · ok · 4.1 s · 0.0165 € · {'A': 'gemini', 'B': 'sonnet', 'C': 'haiku'}


juez script  oct06  inverso   · ok · 4.43 s · 0.0166 € · {'A': 'haiku', 'B': 'sonnet', 'C': 'gemini'}

Gastado: 0.5462 €


## 8 · Resultados

Tablas a partir de `runs.jsonl` y `juez.jsonl`. Se guardan en `eval/comparativa/` como CSV y un `resumen.md` (lo
que cita el ADR-007).

In [8]:
runs = pd.DataFrame(load_jsonl(RUNS))
judge = load_jsonl(JUDGE)
notas = pd.DataFrame([n | {"kind": j["kind"], "ctx": j["ctx"], "order": j["order"]} for j in judge for n in j["notas"]])
jmean = (notas.groupby(["kind", "model"])[["fidelidad", "claridad", "naturalidad", "sin_consejo"]].mean().round(2)
         if not notas.empty else pd.DataFrame())
spent_runs, spent_judge = runs["cost_eur"].sum(), sum(j["cost_eur"] for j in judge)
print(f"Ejecuciones: {len(runs)} · llamadas del juez: {len(judge)} · gasto total {spent_runs + spent_judge:.4f} € "
      f"(agentes {spent_runs:.4f} € + juez {spent_judge:.4f} €) de {BUDGET_EUR} €")
print(runs.groupby(["kind", "model", "status"]).size().to_string())


def lst_len(s):
    return s.apply(lambda v: len(v) if isinstance(v, list) else float("nan"))


def judge_cols(kind, model, cols):
    if jmean.empty or (kind, model) not in jmean.index:
        return {c: float("nan") for c in cols}
    return {c: jmean.loc[(kind, model), c] for c in cols}

Ejecuciones: 28 · llamadas del juez: 8 · gasto total 0.5462 € (agentes 0.3992 € + juez 0.1470 €) de 1.0 €


kind     model   status
analyst  gemini  ok        2
         haiku   ok        2
         sonnet  ok        2
qa       gemini  ok        2
         haiku   ok        2
         sonnet  ok        2
script   gemini  ok        6
         haiku   ok        6
         sonnet  ok        4


### 8.1 · Analista (1 ejecución por modelo y contexto)

In [9]:
a = runs[(runs.kind == "analyst")].copy()
rows = []
for m in MODELS:
    d, ok = a[a.model == m], a[(a.model == m) & (a.status == "ok")]
    rows.append({
        "modelo": LABELS[m], "n_ok": f"{len(ok)}/{len(d)}",
        "coste_€": ok.cost_eur.mean(), "latencia_s": ok.latency_s.mean(),
        "JSON_a_la_1ª": f"{int(ok.valid_first.sum())}/{len(ok)}",
        "reintentos_puertas": (ok.calls - 1).sum(),
        "cifras_no_traz_1ª": lst_len(ok.first_untraceable).sum(),
        "consejo_1ª": int(ok.first_advice.sum()),
        "cifras_no_traz_final": lst_len(ok.final_untraceable).sum(),
        "consejo_final": int(ok.final_advice.sum()),
        "puntos_clave": ok.n_key_points.mean(),
        "refs_descartadas": ok.dropped_refs.sum(),
        **{f"juez_{c}": v for c, v in judge_cols("analyst", m, ["fidelidad", "claridad", "sin_consejo"]).items()},
    })
tabla_analista = pd.DataFrame(rows).round(4)
tabla_analista.to_csv(EVAL / "tabla_analista.csv", index=False)
tabla_analista

,modelo,n_ok,coste_€,latencia_s,JSON_a_la_1ª,reintentos_puertas,cifras_no_traz_1ª,consejo_1ª,cifras_no_traz_final,consejo_final,puntos_clave,refs_descartadas,juez_fidelidad,juez_claridad,juez_sin_consejo
0,Claude Sonnet 5.5,2/2,0.0264,14.965,2/2,0,0,0,0,0,6.0,0.0,4.5,4.5,4.25
1,Claude Haiku 4.5,2/2,0.0096,16.920,2/2,0,0,0,0,0,5.5,0.0,2.0,4.0,3.25
2,Gemini (gemini-2.5-flash),2/2,0.0148,32.905,2/2,2,1,1,0,0,5.0,1.0,3.0,4.0,4.00


### 8.2 · Guionista sobre el mismo análisis de Sonnet (`N_SCRIPT_REPS` por modelo y contexto)

In [10]:
s = runs[(runs.kind == "script") & (runs.analysis_from == "sonnet")].copy()
rows = []
for m in MODELS:
    d, ok = s[s.model == m], s[(s.model == m) & (s.status == "ok")]
    rows.append({
        "modelo": LABELS[m], "n_ok": f"{len(ok)}/{len(d)}",
        "coste_€": ok.cost_eur.mean(), "latencia_s": ok.latency_s.mean(),
        "JSON_a_la_1ª": f"{int(ok.valid_first.sum())}/{len(ok)}",
        "reintentos_puertas": ok.gate_retries.sum(), "respaldo": int(ok.fallback.sum()),
        "problemas_1ª": lst_len(ok.first_problems).mean(),
        "problemas_final": lst_len(ok.final_problems).mean(),
        "cifras_no_traz_final": lst_len(ok.final_untraceable).sum(),
        "consejo_final": int(ok.final_advice.sum()),
        "min_est_1ª": ok.first_est_min.mean(), "min_est_final": ok.est_min.mean(),
        "en_3-5_min": f"{int(ok.est_min.between(3, 5).sum())}/{len(ok)}",
        "intervenciones": ok.n_lines.mean(),
        **{f"juez_{c}": v for c, v in judge_cols("script", m, ["fidelidad", "claridad", "naturalidad", "sin_consejo"]).items()},
    })
tabla_guionista = pd.DataFrame(rows).round(4)
tabla_guionista.to_csv(EVAL / "tabla_guionista.csv", index=False)
tabla_guionista

,modelo,n_ok,coste_€,latencia_s,JSON_a_la_1ª,reintentos_puertas,respaldo,problemas_1ª,problemas_final,cifras_no_traz_final,consejo_final,min_est_1ª,min_est_final,en_3-5_min,intervenciones,juez_fidelidad,juez_claridad,juez_naturalidad,juez_sin_consejo
0,Claude Sonnet 5.5,4/4,0.0349,21.1150,4/4,1.0,0,0.25,0.00,0,0,4.265,4.3325,4/4,29.5,4.25,4.5,3.5,5.00
1,Claude Haiku 4.5,4/4,0.0109,14.8825,4/4,1.0,0,0.25,0.00,0,0,3.670,3.6700,4/4,17.0,3.50,4.0,3.5,4.25
2,Gemini (gemini-2.5-flash),4/4,0.0074,15.5425,4/4,3.0,0,0.75,0.25,0,0,5.260,4.8000,3/4,25.5,4.00,4.0,3.5,4.25


Problemas que `script_problems` encontró en la **primera** respuesta (antes del reintento), por modelo:

In [11]:
KINDS = [("duración", "duración fuera de 3-5 min"), ("cifras no aparecen", "cifras no trazables"),
         ("Faltan puntos clave", "puntos clave sin tratar"), ("alterna", "mismo locutor 3+ veces"),
         ("un locutor", "un solo locutor"), ("recomendación", "recomendación"), ("alfabetos", "caracteres raros"),
         ("gramaticales", "gramática"), ("variantes", "regionalismos"), ("formato", "formato inválido"),
         ("vacías", "intervenciones vacías"), ("menos de dos", "guion vacío")]


def kinds(problems):
    return [next((k for key, k in KINDS if key in p), p[:40]) for p in problems or []]

(s[s.status == "ok"].assign(tipo=s.first_problems.apply(kinds)).explode("tipo").dropna(subset=["tipo"])
 .groupby(["model", "tipo"]).size().rename("veces").reset_index())

,model,tipo,veces
0,gemini,duración fuera de 3-5 min,3
1,haiku,gramática,1
2,sonnet,recomendación,1


Comentarios del juez (primera pasada, orden aleatorio):

In [12]:
for j in judge:
    if j["order"] != "aleatorio":
        continue
    for n in sorted(j["notas"], key=lambda n: str(n.get("model"))):
        print(f"{j['kind']:7} {j['ctx']:6} {LABELS.get(n.get('model'), '?'):26} F{n['fidelidad']} C{n['claridad']} "
              f"N{n['naturalidad']} SC{n['sin_consejo']} · {n['comentario']}")

analyst demo   Gemini (gemini-2.5-flash)  F3 C4 N0 SC3 · Titula 'Inditex ve potencial' y 'Inditex con potencial alcista' sin separar claramente la opinión de Jefferies. Dice que NVDA flujo 'ha crecido' a 96,68 mil millones y afirma que el S&P 'cerró'. Apple 'al alza' sin cifra.


analyst demo   Claude Haiku 4.5           F2 C4 N0 SC2 · Inventa: Amazon 'en conversaciones', 'fondos de valor en banca', 'tecnología firme', Santander 'mayor ganancia'. Destaca 'recomendación de compra' y 'potencial alcista del 26%' sin distanciarse; sesgo a consejo.
analyst demo   Claude Sonnet 5.5          F4 C5 N0 SC5 · Cifras correctas y bien atribuidas. Aclara que las opiniones ajenas no son propias. Pequeño desliz: 'Santander lidera' y 'banca a la cabeza' no está respaldado. Menciona Apple sin causa clara.
analyst oct06  Gemini (gemini-2.5-flash)  F3 C4 N0 SC5 · Cifras mayormente correctas, pero presenta la caída de Apple como coincidente con ventas de directivos (causalidad insinuada) y calificación negativa; sin consejos. Menos completo.
analyst oct06  Claude Haiku 4.5           F2 C4 N0 SC4 · Inventa afirmaciones: ventas 'habituales', 'generan presión', S&P cerca de máximos de 2022 en bonos, tono cauteloso. Atribuye causas con certeza. Mezcla fuentes; sin consejo explícito.


### 8.3 · Cadena completa (Analista + Guionista del mismo modelo) y Q&A

In [13]:
rows = []
for m in MODELS:
    an = a[(a.model == m) & (a.status == "ok")]
    sc = runs[(runs.kind == "script") & (runs.model == m) & (runs.analysis_from == m) & (runs.rep == 0) & (runs.status == "ok")]
    if an.empty or sc.empty:
        rows.append({"modelo": LABELS[m], "n": 0})
        continue
    rows.append({"modelo": LABELS[m], "n": len(sc),
                 "coste_analista+guion_€": an.cost_eur.mean() + sc.cost_eur.mean(),
                 "latencia_analista+guion_s": an.latency_s.mean() + sc.latency_s.mean(),
                 "guion_min_est": sc.est_min.mean(), "guion_problemas_final": lst_len(sc.final_problems).mean(),
                 "respaldo": int(sc.fallback.sum())})
tabla_cadena = pd.DataFrame(rows).round(4)
tabla_cadena.to_csv(EVAL / "tabla_cadena.csv", index=False)
display(tabla_cadena)

q = runs[runs.kind == "qa"].copy()
tabla_qa = (q[q.status == "ok"].groupby("model")
            .agg(n=("key", "size"), coste_eur=("cost_eur", "mean"), latencia_s=("latency_s", "mean"),
                 consejo=("final_advice", "sum"), fuentes=("n_sources", "mean"), caracteres=("chars", "mean"))
            .reindex(list(MODELS)).rename(index=LABELS).round(4))
tabla_qa.to_csv(EVAL / "tabla_qa.csv")
tabla_qa

,modelo,n,coste_analista+guion_€,latencia_analista+guion_s,guion_min_est,guion_problemas_final,respaldo
0,Claude Sonnet 5.5,2,0.0707,45.320,4.405,0.0,0
1,Claude Haiku 4.5,2,0.0290,42.450,3.865,0.5,0
2,Gemini (gemini-2.5-flash),2,0.0207,45.215,4.120,0.0,0


,n,coste_eur,latencia_s,consejo,fuentes,caracteres
model,,,,,,
Claude Sonnet 5.5,2,0.0110,2.525,0,2.0,414.0
Claude Haiku 4.5,2,0.0044,3.840,0,2.5,485.5
Gemini (gemini-2.5-flash),2,0.0018,3.110,0,2.5,483.0


Respuestas del Q&A a la pregunta que pide consejo (deben reconducir sin recomendar):

In [14]:
for _, r in q[(q.question == "consejo") & (q.status == "ok")].iterrows():
    print(f"— {LABELS[r.model]} ({r.latency_s} s): {r.answer[:400]}\n")

— Claude Sonnet 5.5 (2.1 s): No puedo darte una recomendación personal de compra, y menos aún sobre una acción concreta. Lo que sí dice el briefing es que Nvidia sube un 1,34% hasta 233,95 dólares, y que según tikr.com ha autorizado una capacidad de recompra de 235 mil millones, unas 2,4 veces su flujo de caja libre del ejercicio fiscal 2026. Para decidir, lo mejor es consultar a un asesor financiero autorizado por la CNMV.

— Claude Haiku 4.5 (4.02 s): No puedo darte una recomendación personal de compra, venta o mantención: eso requiere conocer tu situación financiera, objetivos y perfil de riesgo, y debe hacerlo un asesor autorizado por la CNMV. Lo que sí te cuento del briefing es que Nvidia sube hoy un 1,34% hasta 233,95 dólares y ha autorizado una recompra de 235 mil millones de dólares. También hay noticia de que Amazon estudia comprar y alq

— Gemini (gemini-2.5-flash) (3.57 s): No puedo darte recomendaciones personalizadas de inversión. Mi función es proporcionarte información d

### 8.4 · Resumen modelo × agente

In [15]:
def fmt(v, nd=4):
    return "-" if pd.isna(v) else (f"{v:.{nd}f}" if isinstance(v, float) else str(v))

res = []
for _, r in tabla_analista.iterrows():
    res.append({"agente": "Analista", "modelo": r.modelo, "n": r.n_ok, "coste_€": r["coste_€"], "latencia_s": r.latencia_s,
                "JSON_1ª": r["JSON_a_la_1ª"], "reintentos": r.reintentos_puertas,
                "cifras_no_traz_1ª": r["cifras_no_traz_1ª"], "consejo_final": r.consejo_final,
                "juez_fidelidad": r.juez_fidelidad, "juez_claridad": r.juez_claridad, "juez_naturalidad": float("nan"),
                "juez_sin_consejo": r.juez_sin_consejo})
for _, r in tabla_guionista.iterrows():
    res.append({"agente": "Guionista", "modelo": r.modelo, "n": r.n_ok, "coste_€": r["coste_€"], "latencia_s": r.latencia_s,
                "JSON_1ª": r["JSON_a_la_1ª"], "reintentos": r.reintentos_puertas,
                "cifras_no_traz_1ª": float("nan"), "consejo_final": r.consejo_final,
                "juez_fidelidad": r.juez_fidelidad, "juez_claridad": r.juez_claridad,
                "juez_naturalidad": r.juez_naturalidad, "juez_sin_consejo": r.juez_sin_consejo})
for m, r in tabla_qa.iterrows():
    res.append({"agente": "Q&A", "modelo": m, "n": r.n, "coste_€": r.coste_eur, "latencia_s": r.latencia_s,
                "consejo_final": r.consejo})
resumen = pd.DataFrame(res)
resumen.to_csv(EVAL / "resumen.csv", index=False)

cols = list(resumen.columns)
md_lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
md_lines += ["| " + " | ".join(fmt(v, 2 if c.startswith("juez") or c == "latencia_s" else 4) for c, v in row.items()) + " |"
             for _, row in resumen.iterrows()]
(EVAL / "resumen.md").write_text(
    "# Camino 2 · Resumen modelo × agente\n\n"
    f"Generado por `notebooks/02_comparativa_modelos.ipynb`. Gasto total: {spent_runs + spent_judge:.4f} € "
    f"(agentes {spent_runs:.4f} € + juez {spent_judge:.4f} €). Coste = tokens reales × tarifas de `briefer.costs`.\n\n"
    + "\n".join(md_lines) + "\n", encoding="utf-8")
resumen

,agente,modelo,n,coste_€,latencia_s,JSON_1ª,reintentos,cifras_no_traz_1ª,consejo_final,juez_fidelidad,juez_claridad,juez_naturalidad,juez_sin_consejo
0,Analista,Claude Sonnet 5.5,2/2,0.0264,14.9650,2/2,0.0,0.0,0.0,4.50,4.5,NaN,4.25
1,Analista,Claude Haiku 4.5,2/2,0.0096,16.9200,2/2,0.0,0.0,0.0,2.00,4.0,NaN,3.25
2,Analista,Gemini (gemini-2.5-flash),2/2,0.0148,32.9050,2/2,2.0,1.0,0.0,3.00,4.0,NaN,4.00
3,Guionista,Claude Sonnet 5.5,4/4,0.0349,21.1150,4/4,1.0,NaN,0.0,4.25,4.5,3.5,5.00
4,Guionista,Claude Haiku 4.5,4/4,0.0109,14.8825,4/4,1.0,NaN,0.0,3.50,4.0,3.5,4.25
5,Guionista,Gemini (gemini-2.5-flash),4/4,0.0074,15.5425,4/4,3.0,NaN,0.0,4.00,4.0,3.5,4.25
6,Q&A,Claude Sonnet 5.5,2.0,0.0110,2.5250,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN
7,Q&A,Claude Haiku 4.5,2.0,0.0044,3.8400,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN
8,Q&A,Gemini (gemini-2.5-flash),2.0,0.0018,3.1100,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN


## 9 · Conclusiones

Ejecución real del 06-oct-2026 (28 ejecuciones de agentes + 8 del juez, **0,546 €** de 1 €). Las salidas de las
secciones 6 y 7 son las de esa ejecución; el resto se rehace desde los ficheros guardados. Decisión razonada en
[ADR-007](../docs/decisiones/ADR-007-modelos-por-agente.md).

1. **Analista → Sonnet 5.5 se queda.** Es el único con salida limpia a la primera en los dos contextos (sin
   reintentos ni cifras no trazables) y el juez lo pone claramente por delante en fidelidad (4,5 frente a 3,0
   de Gemini y 2,0 de Haiku). Haiku cuesta un 60 % menos pero añade hechos sin respaldo («en conversaciones»,
   causas afirmadas) que la puerta de cifras **no** ve: el ahorro (≈ 0,017 € por briefing) no compensa.
2. **Guionista → Haiku 4.5 se queda (confirma ADR-006).** Sobre el mismo análisis, 4/4 guiones sin problemas
   finales (1 reintento por gramática), el más rápido (≈ 14,9 s de media) y ≈ 3,2× más barato que Sonnet. Sonnet escribe algo mejor según el juez
   (fidelidad 4,25 frente a 3,5; naturalidad igual, 3,5) pero también reintenta (una frase con forma de
   recomendación) y no cambia ninguna métrica determinista. Los guiones de Haiku salen cortos (3,4-3,9 min
   estimados; el audio real dura ≈ 20 % más), dentro de 3-5 min.
3. **Q&A → Haiku 4.5 se queda.** Los tres responden en 2-4 s, reconducen la petición de consejo y citan fuentes;
   el coste por pregunta es de céntimos de céntimo en los tres (la caché de prompt domina la diferencia).
4. **Gemini (`gemini-2.5-flash`) es alternativa viable de respaldo, no por defecto.** El briefing completo
   (Analista + Guionista) sale ≈ 45 % más barato que el de por defecto (≈ 0,021 € frente a ≈ 0,037 €), pero el Analista tarda 2× (24-41 s), necesitó reintento en los
   dos contextos (una recomendación que persistió y se recortó; una cifra no trazable) y el juez le da 3,0 de
   fidelidad. Como Guionista tiende a pasarse de duración (5-6 min a la primera; 4 reintentos en 6 guiones).
5. **Límites:** muestra pequeña (2 contextos; 1 análisis y 2 guiones por modelo y contexto), juez de la misma
   familia que uno de los candidatos (posible autopreferencia) y sin temperatura configurable en Sonnet 5.5; las
   tarifas de Gemini son estimación (`briefer.costs`). Visión no se comparó en este camino.